In [135]:
import pandas as pd
import numpy as np

In [136]:
df1 = pd.read_csv('../dataset/raw/regular_season_box_scores_2010_2024_part_1.csv')
df2 = pd.read_csv('../dataset/raw/regular_season_box_scores_2010_2024_part_2.csv')
df3 = pd.read_csv('../dataset/raw/regular_season_box_scores_2010_2024_part_3.csv')
df4 = pd.read_csv('../dataset/raw/regular_season_totals_2010_2024.csv')

df = pd.concat([df1, df2, df3], axis = 0)
df.dropna()
def convert_minutes(x):
    if pd.isna(x):
        return None
    try:
        minutes, seconds = map(int, x.split(':'))
        return minutes + seconds / 60
    except:
        return None

df['minutes'] = df['minutes'].apply(convert_minutes)
team_df = pd.read_csv('../dataset/raw/regular_season_totals_2010_2024.csv')
player_anomaly = pd.read_csv('../dataset/clean/players_with_anomaly.csv')

In [137]:
team_df_sorted = team_df.sort_values(by='GAME_ID')

merged = team_df_sorted.merge(
    team_df_sorted,
    on='GAME_ID',
    suffixes=('', '_opponent'),
    how='inner'
)
merged = merged[merged['TEAM_ID'] != merged['TEAM_ID_opponent']]

merged['PLUS_MINUS'] = merged['PTS'] - merged['PTS_opponent']

result = merged[['GAME_ID', 'PTS', 'PTS_opponent', 'PLUS_MINUS']]
result.rename(columns={'GAME_ID': 'gameId'}, inplace=True)

new_df = df.merge(result, on = 'gameId', how='inner')
new_anomaly = player_anomaly.merge(result, on = 'gameId', how='inner')

C:\Users\Raposo\AppData\Local\Temp\ipykernel_22640\2062307131.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result.rename(columns={'GAME_ID': 'gameId'}, inplace=True)


In [138]:
new_anomaly.columns

Index(['Unnamed: 0', 'personName', 'gameId', 'season_year', 'position',
       'teamId', 'personId', 'minutes', 'fieldGoalsMade',
       'fieldGoalsAttempted', 'fieldGoalsPercentage', 'threePointersMade',
       'threePointersAttempted', 'threePointersPercentage', 'freeThrowsMade',
       'freeThrowsAttempted', 'freeThrowsPercentage', 'reboundsOffensive',
       'reboundsDefensive', 'reboundsTotal', 'assists', 'steals', 'blocks',
       'turnovers', 'foulsPersonal', 'points', 'plusMinusPoints',
       'anomaly_score', 'anomaly_flag', 'seasonMeanMinutes', 'PTS',
       'PTS_opponent', 'PLUS_MINUS'],
      dtype='object')

In [139]:
new_anomaly.rename(columns={'seasonMeanMinutes': 'WS'}, inplace=True)

In [140]:
top20 = new_anomaly.groupby(by = 'gameId').first().sort_values(by = 'WS', ascending = False).reset_index()
top20_g = top20[(top20['position'] == 'G') & (top20['season_year'] == '2022-23')].drop_duplicates(subset='personName', keep='first')
top20_f = top20[(top20['position'] == 'F') & (top20['season_year'] == '2022-23')].drop_duplicates(subset='personName', keep='first')
top20_c = top20[(top20['position'] == 'C') & (top20['season_year'] == '2022-23')].drop_duplicates(subset='personName', keep='first')
player_name = list(top20[top20['season_year'] == '2022-23']['personName'])
main_columns = top20[['teamId', 'PLUS_MINUS', 'gameId']]

In [141]:
teste = df.merge(main_columns, on = ['teamId', 'gameId'])

In [142]:
positions = ['C', 'F', 'G']

for pos in positions:
    top20_p = top20[top20['position'] == pos]
    player_names = top20_p[top20_p['anomaly_flag'] == -1]['personName'].drop_duplicates().head(20).tolist()

    results = []

    for player in player_names:
        player_stat = teste[
            (teste['personName'] == player) & 
            (teste['season_year'] == '2022-23')
        ]

        jogos_com_anomalia = top20_p[
            (top20_p['anomaly_flag'] == -1) & 
            (top20_p['personName'] == player)
        ]['PLUS_MINUS'].mean()

        jogos_sem_anomalia = player_stat[
            ~player_stat['gameId'].isin(
                top20_p[
                    (top20_p['anomaly_flag'] == -1) & 
                    (top20_p['personName'] == player)
                ]['gameId']
            )
        ]['PLUS_MINUS'].mean()

        results.append({
            'Jogador': player,
            'Plus-Minus (com anomalia)': jogos_com_anomalia,
            'Plus-Minus (sem anomalia)': jogos_sem_anomalia,
        })

    df_results = pd.DataFrame(results)
    print(f'\nResultados para posição {pos}:\n')
    print(df_results)

    df_results.to_csv(f'plus_minus_{pos}_2022-23.csv', index=False)


Resultados para posição C:

               Jogador  Plus-Minus (com anomalia)  Plus-Minus (sem anomalia)
0        Anthony Davis                       14.0                 -18.666667
1        Deandre Ayton                      -13.0                   0.375000
2          Rudy Gobert                        1.0                   3.000000
3        Jarrett Allen                        2.0                   2.666667
4    Jonas Valanciunas                       -1.5                  -2.500000
5         Jusuf Nurkic                      -10.0                  -1.200000
6          Jalen Smith                       -3.0                  -1.200000
7   Aleksej Pokusevski                       -1.5                  -5.000000
8         Santi Aldama                        7.0                 -18.666667
9          Bam Adebayo                      -13.0                  -4.250000
10       Dwight Powell                        3.0                   2.000000
11       Blake Griffin                       -6